In [25]:
from pathlib import Path
from pyteomics import mzid
import json
import re

In [9]:
SAMPLE_ACCESSION = "PXD001357"
root_path = Path.cwd() / "project_archive"
project_path = root_path / SAMPLE_ACCESSION

filepath = project_path / "OTE0019_York_060813_JH16_F119502.mzid"

In [ ]:
obj = mzid.MzIdentML(source=str(filepath))

In [ ]:
from typing import Any


counter = 0
for item in mzid.read(source=str(filepath)):
    counter += 1

    # if item.get("spectrumID") == "index=3809":
        # print(item)
        # print(json.dumps(item, indent=4))

    title = item.get("spectrum title")
    pattern = re.compile(r'File:"(?P<filename>[^"]+)"[\s\S]*?scan=(?P<scan>\d+)')
    m = pattern.search(title)
    if not m:
        print(f"WARNING: No match for {item.get('spectrumID')} with title {title}")
        continue

    filename = Path(m.group('filename')).stem
    scan_number = int(m.group('scan'))

    if len(item["SpectrumIdentificationItem"]) > 1:
        print(f"WARNING: Multiple SpectrumIdentificationItem for {item.get('spectrumID')}")

    seq_obj: dict[str, Any] = item["SpectrumIdentificationItem"][0]
    charge = seq_obj.get("chargeState")
    seq: list[str] = seq_obj.get("PeptideSequence", "").split()
    if seq_obj.get("Modification"):
        inserted = 0
        for mod in seq_obj["Modification"]:
            loc = mod.get("location") + inserted
            if loc > 1:
                loc -= 1  # shift for 1-based indexing
            name = mod.get("name")
            # insert modification notation into sequence
            seq.insert(loc, f"[{name}]")
            inserted += 1

        usi = f"mzspec:{SAMPLE_ACCESSION}:{filename}:scan:{scan_number}:{''.join(seq)}/{charge}"

        if len(seq_obj.get("Modification")) > 1:
            print(f"raw sequence: {seq_obj.get('PeptideSequence')}")
            print(json.dumps(seq_obj.get("Modification"), indent=4))
            print(usi)

print(f"Total items: {counter}")